# NN 이진분류
- 피마 인디언 당뇨병 예측 데이터셋을 이용해서 이진분류를 실시

In [ ]:
!pip install ipython-autotime
%load_ext autotime

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras

SEED = 42

## 1. 데이터 준비

In [ ]:
path = 'https://raw.githubusercontent.com/20161609/data_box/refs/heads/main/diabetes.csv'
diabetes = pd.read_csv(path)
diabetes.shape

In [ ]:
diabetes.head()

In [ ]:
df = diabetes.copy()
df.info() # -> 결측치 없음, 전부 수치형 (인코딩 필요x)

In [ ]:
df.describe().T

### 범주형 변수

In [ ]:
df.columns

In [ ]:
df['Outcome'].value_counts()

In [ ]:
sns.countplot(data=df, x='Outcome')

### 연속형 변수

In [ ]:
tmp = df['Pregnancies'].sort_values(ascending=False)
tmp = tmp.reset_index()
tmp.head()

In [ ]:
sns.barplot(x=tmp.index, y = tmp['Pregnancies'])

In [ ]:
df.hist()

### 결측치

In [ ]:
df.isna().sum()

### 중복치

In [ ]:
df.duplicated().sum()

### 이상치

In [ ]:
# 박스플롯 그리기

In [ ]:
df.describe().T

## 2. 트레인, 테스트 데이터 분리

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.1, random_state=SEED, stratify=df['Outcome'])
train.shape, test.shape

In [ ]:
train['Outcome'].value_counts()

In [ ]:
train.head()

### X, y 변수 분리

In [ ]:
X_train = train.drop('Outcome', axis=1)
y_train = train['Outcome']

X_train.shape, y_train.shape

In [ ]:
# 이상치 0인 값을 특정 값(중간 값)으로 치환
# 'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'

In [ ]:
median_list = []

col_list = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in col_list:
  med = X_train[col].median()
  X_train.loc[X_train[col] == 0, col] = med
  median_list.append(med)

In [ ]:
X_train.describe().T # min값이 0인 값이 없음을 확인

### 스케일링

In [ ]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
X_train_s = ss.fit_transform(X_train)
X_train_s # np.array 로 자동으로 바뀜

In [ ]:
print(ss.mean_) # 각 컬럼당 평균값
print(ss.var_) # 각 컬럼당 분산값

In [ ]:
y_train_e = y_train.to_numpy()
y_train_e

In [ ]:
print(X_train_s.shape, y_train_e.shape)
print(type(X_train_s), type(y_train_e))

## 모델 학습

In [ ]:
from keras import layers

model = keras.Sequential([
    layers.Dense(8, activation='relu', input_shape=(8,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid') # 이진분류는 시그모이드로 가라 -> 확률로 나옴
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
EPOCHS = 100
BATCH_SIZE = 16

history = model.fit(
    X_train_s, y_train_e,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2
)

In [ ]:
def plot_history(history):
    hist = pd.DataFrame(history.history)
    hist['epoch'] = history.epoch

    plt.figure(figsize=(16, 8))
    plt.subplot(1, 2, 1)
    plt.xlabel('epochs')
    plt.ylabel('loss')
    plt.plot(hist['epoch'], hist['loss'], label='train loss')
    plt.plot(hist['epoch'], hist['val_loss'], label='val loss')
    plt.title('Loss Curve')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.xlabel('epochs')
    plt.ylabel('accuracy')
    plt.plot(hist['epoch'], hist['accuracy'], label='train accuracy')
    plt.plot(hist['epoch'], hist['val_accuracy'], label='val accuracy')
    plt.title('Accuracy Curve')
    plt.legend()
    plt.show()

In [ ]:
plot_history(history)

In [ ]:
X_test = test.drop('Outcome', axis=1)
y_test = test['Outcome']

X_test.shape, y_test.shape

In [ ]:
# 테스트값 전처리
# median_list = []
col_list = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for i, col in enumerate(col_list):
  X_test.loc[X_test[col] == 0, col] = median_list[i]
  median_list.append(med)

In [ ]:
X_test_s = ss.transform(X_test)
X_test_s

In [ ]:
y_test_e = y_test.to_numpy()
y_test_e

In [ ]:
y_pred = model.predict(X_test_s)
y_pred = (y_pred > 0.5).astype(int).reshape(-1)
y_pred.shape

In [ ]:
y_test_e

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.metrics import confusion_matrix

def print_metrics(y_true, y_pred, ave='binary'):
  print('accuracy:', accuracy_score(y_test_e, y_pred))
  print('recall:', recall_score(y_test_e, y_pred, average=ave))
  print('precision:', precision_score(y_test_e, y_pred, average=ave))
  print('f1 :', f1_score(y_test_e, y_pred, average=ave))

  clm = confusion_matrix(y_test_e, y_pred)
  s = sns.heatmap(clm, annot=True, fmt='d', cbar=False)
  s.set(xlabel='Predicted', ylabel='Actual')
  plt.show()


In [ ]:
print_metrics(y_test_e, y_pred)